# Exercise 3: Floating-Base Kinematics

In Exercise 2, we investigated the kinematics of the Tiago robot assuming its base was fixed. We now extend the analysis to leverage its floating-base nature and, e.g., follow targets that are initially out of reach. This notebook covers:
- Understanding of motion constraints for wheeled locomotion
- Implementation of floating-base forward and inverse kinematics
- Whole-body control with weighted distribution of control effort through numerical, weighted inverse kinematics.

## Learning Goals
1. Derive constrained equations for a 90° Swedish wheel.
2. Formulate forward and inverse kinematics of a multi-wheel mobile base.
3. Implement a whole-body inverse kinematics solver.
4. Reach and track targets using the combined motion of the wheeled base and arm.


In [ ]:
import time
import numpy as np
import pinocchio as pin
import example_robot_data as RobData
from pinocchio.visualize import MeshcatVisualizer
import utils as Utils

ENABLE_VIEWER = True

## 1) Setup Robot and Viewer
After running the following code block, a web based visualization window should pop up, showing a tiago robot in home configuration.


TIPS: It's recommended to put code and visualization side by side to see the full process of visualization.

In [ ]:
robot = RobData.load("tiago")
model = robot.model
data = model.createData()
theta_home = robot.q0.copy() # Note: AMR lecture use 'theta' for joint configuration, Pinnochio 'q'

if False: # enable if you want to look up joint/frame IDs
    print("joints:")
    for i, n in enumerate(model.names):
        print(i, n)

    print("frames:")
    for i, n in enumerate(model.frames):
        print(i, n.name)

B_FRAME_ID = robot.model.getFrameId('universe')       # ID of link of the base (termed universe in Pinocchio since it doesn't consider floating/mobile bases)
E_FRAME_ID = robot.model.getFrameId("hand_palm_link") # ID of link termed end-effector E in this exercise

viz = None
viewer_enabled = False
if ENABLE_VIEWER:
    viz = MeshcatVisualizer(model, robot.collision_model, robot.visual_model)
    try:
        viz.initViewer(open=True)
        viz.loadViewerModel("tiago")
        viewer_enabled = True
    except Exception as e:
        print(f"[warning] Meshcat unavailable, fallback to headless: {e}")

if viewer_enabled:
    viz.display(theta_home)

## 2) Recap Notation and Kinematics Computation with Pinocchio

[NOTE] AMR lecture uses $\bm{\theta}$ as joint configuration vector (joint states) vs. Pinocchio using $\mathbf{q}$.

Recall, in order to evaluate the forward kinematics, first update the kinematic model using a given joint configuration, e.g. theta_home. After updating the kinematics model, you can query any frame via its ID. For example, we can query the pose of the end-effector or base, as well as the end-effector Jacobian. To simplify retrieving the pose of a link, below we defined a small helper function that combines the update & call.

[NOTE] Pinocchio can only return kinematic quantities w.r.t base, but not the world once we consider a floating base where world-frame W $\not=$ base-frame B. The pose of the base frame as queried below, will therefore always be considered at the origin by Pinocchio. Pinocchio is not aware of wheel-related motion constraints (not the ones in the URDF, let alone the custom ones considered in this exercise below), it assumes that they spin in free air and don't affect the kinematics. Therefore, the wheel-related components and transformations of the base need to be coded-up from scratch, i.e. without the Pinocchio API. 

In [ ]:
pin.framesForwardKinematics(robot.model, robot.data, theta_home) 
pin.updateFramePlacements(model, data) # ! always neccessary to update the frame 

J_E = pin.computeFrameJacobian(robot.model, robot.data, theta_home, E_FRAME_ID, pin.ReferenceFrame.LOCAL_WORLD_ALIGNED)

print("end-effector pose wrt origin of fixed-base:", robot.data.oMf[E_FRAME_ID])
print("base pose wrt origin of fixed-base:", robot.data.oMf[B_FRAME_ID])
print(f'shape of end-effector Jacobian: {J_E.shape}')

def get_SE3_pose(model: pin.Model, data: pin.Data, theta: np.ndarray, e_frame_id: int) -> np.ndarray:
    pin.forwardKinematics(model, data, theta)
    pin.updateFramePlacements(model, data)
    
    return data.oMf[e_frame_id].homogeneous.copy()

## 3) Floating-Base Forward Kinematics 

### Task 1.3
Implement the following function to return the wheel constraints $\mathbf{c}$ for a 90 degree swedish wheel with parametrization as indicated in Fig. 2 in the exercise pdf.

Input:  wheel parameters as dict with keys 'alpha', 'beta', 'l', and 'r', e.g. ```wheel = {'alpha': 0.5, 'beta': 0.1, 'l': 0.25, 'r': 0.1} ```

Output:  $\mathbf{c}$ s.t. $\mathbf{c} {}_B\bm{\nu}_{WB} = \omega$ where ${}_B\bm{\nu}_{WB} = [{}_B\text{v}_{WB,x},{}_B\text{v}_{WB,y},\Omega_{WB}]^T$ is the base-twist expressed in robot base frame and $\omega$ is the wheel rotational velocity.

In [ ]:
def get_wheel_constraint_jacobian(wheel) -> np.ndarray:
    # Returns a 1x3 numpy array c that projects the 2D floating-base twist [v_fwd, v_side, Omega]^T
    # onto the wheel velocity omega, i.e. c@[v_fwd, v_side, Omega]^T = omega
    # c expresses the no-slip and rolling constraints that need to hold for any valid motion of the base

    # ===== TODO =====
    return np.array([np.sin(wheel['alpha'] + wheel['beta']), -np.cos(wheel['alpha']+wheel['beta']), -np.cos(wheel['beta'])*wheel['l']])/wheel['r']
    # ===== END TODO =====

    # print("get_wheel_constraint_jacobian() not implemented")
    # return np.zeros((3,1))

# test
wheel = {'alpha': 0.5, 'beta': 0.1, 'l': 0.25, 'r': 0.1}
print(get_wheel_constraint_jacobian(wheel) - np.array([5.64642473, -8.25335615, -2.48751041])) # should be close to zero

Next, you'll implement the functionality to calculate the end-effector (E) position Jacobian $\mathbf{J}_{E}$ with respect to both the joints and wheels, expressed in robot base-frame B

${}_B\mathbf{v}_{WE} = \mathbf{J}_{E}(\bm{\theta}) \left[ \begin{matrix} \dot{\bm{\theta}} \\ {\mathbf{u}} \end{matrix} \right] $

To simplify IK, we ignore the wheels specified in the URDF (non-holonomic diff-drive) and instead imagine our own holonomic base with three 90 degree swedish wheels as illustrated in Fig. 2 of the exercise. That's why the state derivative is augmented by the three wheel speeds as written above.

Note that pinocchio can only return kinematic quantities w.r.t base, but not the world once we consider a floating base where world-frame W != base-frame B. Pinocchio is not aware of wheel-related motion constraints (not the ones in the URDF, let alone the custom ones considered here), it assumes they spin in free air and don't affect the kinematics. Therefore, the wheel-related components need to be coded-up from scratch, i.e. without pinocchio API. 

### Task 2.1

First, define the wheel layout as shown in Fig. 2 of the exercise. 

In [ ]:
# Specify the wheel layout for the robot-base as shown in Figure 1
# ===== TODO =====
wheels = [{'alpha': 0, 'beta': 0, 'l': 0.25, 'r': 0.1},
          {'alpha': np.pi*2/3, 'beta': 0, 'l': 0.25, 'r': 0.1},
          {'alpha': np.pi*4/3, 'beta': 0, 'l': 0.25, 'r': 0.1}]
# ===== END TODO =====

### Task 2.2
Then implement ```get_floating_base_jacobian``` to return the Jacobian $\mathbf{J}_{B}$ of the base, relating base twist expressed in base-frame and wheel speeds, i.e. 

${}_B\bm{\nu} = \mathbf{J}_{B}\left[ \begin{matrix} \omega_0 \\ \omega_1 \\ \omega_2 \end{matrix} \right]$

Reuse the function ```get_wheel_constraint_jacobian``` you implemented before. Use a function like np.linalg.inv(), to get the inverse of a matrix.

In [ ]:
def get_floating_base_jacobian(
    wheels: list) -> np.ndarray:
    
    # ===== TODO =====
    # inverse of pose Jacobian of base
    J_B_inv = np.zeros((len(wheels),3))
    for i, w in enumerate(wheels):
        J_B_inv[i,:] = get_wheel_constraint_jacobian(w)

    try:
        return  np.linalg.inv(J_B_inv)
    
    except Exception as e:
        print(f"[warning] {e}, ignoring base kinematics")
        return  np.zeros((3, len(wheels)))   
    # ===== END TODO =====
    
    # print("get_floating_base_jacobian() not implemented")
    # return  np.zeros((3, len(wheels)))  

### Task 2.3

Complete the following helper function to serve as a single discrete integration step. It takes a current pose (w.r.t. world frame), constant wheel speed and (small) time-step, and returns the resulting next pose (w.r.t. world frame). By sequentially calling this function, we can forward integrate the base kinematics. Mind the transformation between world- and base frames when using ```get_floating_base_jacobian```, which is expressed in base frame.

In [ ]:
def step_base (
    wheels: list,
    u: np.ndarray,  
    W_pose: dict = {'pos': np.zeros(2), 'yaw': 0.0},
    dt: float = 0.01) -> np.ndarray:

    # ===== TODO =====
    R_WB = np.array([[np.cos(W_pose['yaw']), -np.sin(W_pose['yaw'])],
                     [np.sin(W_pose['yaw']),  np.cos(W_pose['yaw'])]])

    B_dpose = get_floating_base_jacobian(wheels) @ u * dt
    W_pose['pos'] += (R_WB @ B_dpose[:2])
    W_pose['yaw'] += B_dpose[2]
    # ===== END TODO =====

    return W_pose.copy()

Now we have the basic plumbing to move the robot around. Pick a set of wheel speeds u and see the base translating and rotating. Can you make it move in only x? What speeds to pick so that it turns on the spot?

In [ ]:
# Pick wheel speeds
u = np.array([0,3,-3])

# move around
W_pose_start = {'pos': np.zeros(2), 'yaw': 0.0}
W_pose_next = W_pose_start
for _ in range(100):
    W_pose_next = step_base(wheels, u, W_pose_next, 0.05)
    if viewer_enabled:
        viz.display(theta_home)
        Utils.apply_base_transform(viz, W_pose_next['pos'][0], W_pose_next['pos'][1], W_pose_next['yaw']) 
        time.sleep(0.05)

### Task 2.4 (Advanced)

Finally, with the Jacobian of the base, we can now move on to calculate the end-effector Jacobian. 

Hint, first determine the matrix $\mathbf{A}(\bm{\theta})$ which, for a 'frozen' arm $\dot{\bm{\theta}}=0$, relates the base-twist and end-effector velocity as ${}_B\mathbf{v}_{WE} = \mathbf{A}(\bm{\theta}) {}_B\bm{\nu}_{WB}$, then combine $\mathbf{A}(\bm{\theta})$ and $\mathbf{J}_{B}$ together with the (base-relative) position Jacobian of the end-effector (from Pinocchio, with respect to joints $\bm{\theta}$) to obtain $_{B}\mathbf{J}_{E}(\bm{\theta})$

In [ ]:
def get_floating_base_ee_jacobian(
    model: pin.Model,
    data: pin.Data,
    theta: np.ndarray,  
    e_frame_id: int,
    wheels: list) -> np.ndarray:

    # end-effector position jacobian
    J_E_joints = pin.computeFrameJacobian(model, data, theta, e_frame_id, pin.ReferenceFrame.LOCAL_WORLD_ALIGNED)[:3, :]

    # ===== TODO =====
    # base-twist to ee velocity, expressed in base frame
    J_B = get_floating_base_jacobian(wheels)
    B_pose_BE = get_SE3_pose(model, data, theta, e_frame_id)
    B_pos_BE = B_pose_BE[:3, 3]
    A = np.array([[1, 0, -B_pos_BE[1]], [0, 1, B_pos_BE[0]], [0, 0, 0]])
    J_E_wheels = A@J_B
    # ===== END TODO =====
    
    return np.block([J_E_joints, J_E_wheels])

    # print("get_floating_base_ee_jacobian() not implemented")
    # return np.block([J_joints, np.zeros((3,3))])


## 4) Floating-Base Inverse Kinematics (Advanced)

### Task 3.1 (Advanced)
Similar to Exercise 2, we will apply an iterative inverse kinematics approach to track a moving target. This time, however, we have a moving base and can therefore also approach targets that are initially out of reach. Make use of $\mathbf{J}_{E}(\bm{\theta})$ derived before, and implement the missing parts of ```solve_numerical_ik_pos```

In [ ]:
def solve_numerical_ik_pos(
    model: pin.Model,
    data: pin.Data,
    e_frame_id: int,
    target_pos: np.ndarray,
    theta_init: np.ndarray,
    W_pose_base_init: dict = {'pos': np.zeros(2), 'yaw': 0.0},
    max_iters: int = 120,
    kp: float = 2.0,
    tol: float = 1e-3,
    dt: float = 0.08,
):
    
    theta = theta_init.copy()                 # initial guess for joint angles
    W_pose_base = W_pose_base_init    # initial guess for pose of base w.r.t. world, expressed in world
    for _ in range(max_iters):
        B_pos_BE = get_SE3_pose(model, data, theta, e_frame_id)[:3, 3]

        # ===== TODO =====
        # Formulate end-effector position error "err" expressed in robot base frame. 
        # Use W_pose_base to figure out transforms. Note that B_pos_BE is the end-effector E
        # position expressed in, and w.r.t, the robot-base frame B since pinocchio doesn't
        # know about the wheels.
        R_WB = np.array([[np.cos(W_pose_base['yaw']), -np.sin(W_pose_base['yaw']), 0],
                         [np.sin(W_pose_base['yaw']),  np.cos(W_pose_base['yaw']),  0], 
                         [0,                        0,                         1]])
        R_BW = R_WB.transpose()
        err = R_BW@(target_pos - np.block([W_pose_base['pos'],0])) - B_pos_BE
        # ===== END TODO =====
        
        # Cartesian position error, if error norm < tolerance, exit the loop
        if np.linalg.norm(err) < tol:
            break

        # Perform iterative inverse kinematics update using weighted pseudo-inverse
        # to trade-off motion of base, the vertical column/torso, and the arm joints
                     
        J_E = get_floating_base_ee_jacobian(model, data, theta, e_frame_id, wheels)

        # CHOOSE BETWEEN STANDARD AND WEIGHTED PSEUDOINVERSE
        if True:
            # standard pseudo-inverse
            dtheta_wb = J_E.transpose()@ np.linalg.inv((J_E@J_E.transpose())) @ (kp * err) # increment in joint and wheel angles
            
        else:
            # weighted pseudo-inverse
            W = np.eye(J_E.shape[1])*0.001          # default penalty
            W[-3:,-3:] = 0.01*np.diag([1, 1, 1])    # base penalty
            W[0,0] = 1                              # torso lift penalty
            dtheta_wb = np.linalg.inv(J_E.T@J_E + W)@ J_E.T @ (kp * err) # increment in joint and wheel angles

        # integrate joints
        dtheta_joints = np.clip(dtheta_wb[:-3], -0.1, 0.1)  # increment in joint angles
        theta = pin.integrate(model, theta, dtheta_joints * dt)

        # enforce joint limits
        for i in range(model.lowerPositionLimit.size):
            theta[i] = min(max(theta[i], model.lowerPositionLimit[i]), model.upperPositionLimit[i])

        # integrate base
        u = np.clip(dtheta_wb[-3:], -0.5, 0.5)  # increment in wheel angles
        W_pose_base = step_base(wheels, u, W_pose_base, dt) 

    R_BW = np.array([[np.cos(W_pose_base['yaw']), np.sin(W_pose_base['yaw']), 0], [-np.sin(W_pose_base['yaw']), np.cos(W_pose_base['yaw']),  0],  [0, 0, 1]])
    final_err = np.linalg.norm( R_BW@(target_pos - np.block([W_pose_base['pos'],0])) - get_SE3_pose(model, data, theta, e_frame_id)[:3, 3])
    
    return theta, final_err, W_pose_base

Now test the numerical solver on one static target.

Learning check:
- If convergence is good, position error is usually around `1e-3` m.
- If error is large, try tuning `kp`, `dt`, or `max_iters`.


In [ ]:
target_pos = np.array([0.55, 0.15, 0.35], dtype=float)

Utils.create_sphere(viz, "world/target", radius=0.04, color_hex=0x8fce00)
if viewer_enabled:
    Utils.set_sphere_xyz(viz, "world/target", target_pos)

theta_static, final_err, W_pose_base = solve_numerical_ik_pos(
    model=model,
    data=data,
    e_frame_id=E_FRAME_ID,
    target_pos=target_pos,
    theta_init=theta_home.copy(),
    W_pose_base_init={'pos': np.zeros(2), 'yaw': 0.0},
    max_iters=300,
)
assert theta_static.size != 0, "Analytical IK failed for static target"

if viewer_enabled:
    viz.display(theta_static)
    Utils.apply_base_transform(viz, W_pose_base['pos'][0], W_pose_base['pos'][1], W_pose_base['yaw']) 

## 5) Whole-Body Tracking of an Moving Target (Advanced)
Similarly to Exercise 2, we run numerical IK repeatedly to track a reference trajectory. This time however, the target position is initially out of reach and Tiago needs to use its mobile base to get close.


In [ ]:
traj = Utils.draw_eight_3D_trajectory(size=0.4, center=[1.6, 0.0, 0.65])

theta_num = theta_home.copy()
W_pose_base = {'pos': np.zeros(2), 'yaw': 0.0}

q_results = []
pos_results = []
track_errors = []
yaw_results = []

for p in traj:
    # solve joint configuration for each waypoint
    
    theta_num, err, W_pose_base = solve_numerical_ik_pos(
        model=model,
        data=data,
        e_frame_id=E_FRAME_ID,
        target_pos=p,
        theta_init=theta_num,   # warm-start joint angles from previous solution
        W_pose_base_init=W_pose_base,  # warm-start base pose from previous solution
        max_iters=40,
        dt=0.08,
        kp=1.0,
    )

    if viewer_enabled:
        Utils.set_sphere_xyz(viz, "world/target", p)
        viz.display(theta_num)
        Utils.apply_base_transform(viz, W_pose_base['pos'][0], W_pose_base['pos'][1], W_pose_base['yaw']) 
        time.sleep(0.04)